# 01. Recuperación y división de las particiones por paciente

**Proyecto:** clasificación binaria de imágenes de tomosíntesis digital de mama (BCS-DBT).

Este cuaderno parte de **dos archivos históricos**: `split_train.csv` y `split_test_inicial.csv`. La información aportada sobre el procedimiento original indica que primero se separó aproximadamente 80 % / 20 % y posteriormente se dividió el conjunto reservado en validación y prueba.

**Limitación importante:** no se conserva `split_val.csv` ni el código exacto con que se realizó la segunda división. Aquí se utiliza `GroupShuffleSplit` con semilla 42 para elaborar una **reconstrucción propuesta**, que **no se debe presentar como equivalente a los conjuntos originales de los experimentos** sin evidencia adicional. Los CSV de origen se conservan intactos.

El procedimiento trabaja solo con metadatos; no descarga imágenes DICOM.

## 1. Carga de archivos y configuración

Colocar los CSV en la misma carpeta que este notebook y ejecutar desde esa carpeta en VS Code o Jupyter. En Colab, subir ambos CSV al entorno antes de ejecutar.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEMILLA = 42
CARPETA = Path.cwd()
ARCHIVO_TRAIN = CARPETA / "split_train.csv"
ARCHIVO_RESERVADO = CARPETA / "split_test_inicial.csv"

for archivo in (ARCHIVO_TRAIN, ARCHIVO_RESERVADO):
    if not archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró {archivo.name}. Colocá los CSV en {CARPETA} "
            "o modificá CARPETA para indicar su ubicación."
        )

train = pd.read_csv(ARCHIVO_TRAIN, dtype={"PatientID": "string", "SeriesInstanceUID": "string"})
reservado = pd.read_csv(ARCHIVO_RESERVADO, dtype={"PatientID": "string", "SeriesInstanceUID": "string"})
print("Entrenamiento histórico:", len(train), "series")
print("Conjunto reservado histórico:", len(reservado), "series")

## 2. Validación de los archivos históricos

In [ ]:
columnas = {"PatientID", "StudyUID", "SeriesInstanceUID", "label", "Normal", "Actionable", "Benign", "Cancer"}
for nombre, df in (("train", train), ("reservado", reservado)):
    faltantes = columnas - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas en {nombre}: {sorted(faltantes)}")
    if df[list(columnas)].isna().any().any():
        raise ValueError(f"Hay valores faltantes en columnas esenciales de {nombre}")
    if df["SeriesInstanceUID"].duplicated().any():
        raise ValueError(f"Hay SeriesInstanceUID duplicadas en {nombre}")
    if not df["label"].isin([0, 1]).all():
        raise ValueError(f"Hay etiquetas no binarias en {nombre}")
    categorias = df[["Normal", "Actionable", "Benign", "Cancer"]]
    if not categorias.isin([0, 1]).all().all() or not categorias.sum(axis=1).eq(1).all():
        raise ValueError(f"Etiquetas diagnósticas inconsistentes en {nombre}")
    if not (df["label"] == 1 - df["Normal"]).all():
        raise ValueError(f"La etiqueta binaria no corresponde a la categoría diagnóstica en {nombre}")

pac_train = set(train["PatientID"])
pac_reservado = set(reservado["PatientID"])
solapamiento = pac_train & pac_reservado
assert not solapamiento, f"Pacientes compartidos entre train y reservado: {len(solapamiento)}"

print("Pacientes compartidos entre entrenamiento y reservado:", len(solapamiento))
for nombre, df in (("Entrenamiento", train), ("Reservado", reservado)):
    print(f"{nombre}: {len(df)} series; {df['PatientID'].nunique()} pacientes; etiquetas {df['label'].value_counts().sort_index().to_dict()}")

## 3. Segunda división reconstruida: validación y prueba

Se utiliza una separación por grupos de pacientes de aproximadamente 50 % / 50 % dentro del conjunto reservado. La proporción exacta de **series** puede no ser 50 % / 50 % porque cada paciente puede aportar varias series. **No se afirma que esta llamada replique el algoritmo histórico**, que no fue recuperado.

In [ ]:
divisor = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEMILLA)
indices_val, indices_test = next(
    divisor.split(reservado, y=reservado["label"], groups=reservado["PatientID"])
)
val_reconstruido = reservado.iloc[indices_val].copy().reset_index(drop=True)
test_reconstruido = reservado.iloc[indices_test].copy().reset_index(drop=True)

print("Validación reconstruida:", len(val_reconstruido), "series")
print("Prueba reconstruida:", len(test_reconstruido), "series")

## 4. Verificación de independencia y conservación de series

In [ ]:
particiones = {
    "train_historico": train,
    "val_reconstruido": val_reconstruido,
    "test_reconstruido": test_reconstruido,
}
for nombre_a, a in particiones.items():
    for nombre_b, b in particiones.items():
        if nombre_a >= nombre_b:
            continue
        comunes = set(a["PatientID"]) & set(b["PatientID"])
        assert not comunes, f"Solapamiento de pacientes: {nombre_a} y {nombre_b}"
        print(f"{nombre_a} / {nombre_b}: {len(comunes)} pacientes compartidos")

assert set(val_reconstruido["SeriesInstanceUID"]).isdisjoint(test_reconstruido["SeriesInstanceUID"])
assert set(val_reconstruido["SeriesInstanceUID"]) | set(test_reconstruido["SeriesInstanceUID"]) == set(reservado["SeriesInstanceUID"])

resumen = pd.DataFrame([
    {
        "particion": nombre,
        "series": len(df),
        "pacientes": df["PatientID"].nunique(),
        "negativas": int((df["label"] == 0).sum()),
        "positivas": int((df["label"] == 1).sum()),
    }
    for nombre, df in particiones.items()
])
print(resumen.to_string(index=False))

## 5. Exportación de la propuesta reconstruida

Los nombres incluyen la palabra **reconstruido** para no confundir estos archivos con los que produjeron los resultados de la tesis. Los CSV históricos no se sobrescriben.

In [ ]:
SALIDA = CARPETA / "salidas_reconstruccion"
SALIDA.mkdir(exist_ok=True)
val_reconstruido.to_csv(SALIDA / "split_val_reconstruido.csv", index=False)
test_reconstruido.to_csv(SALIDA / "split_test_reconstruido.csv", index=False)
resumen.to_csv(SALIDA / "resumen_particiones_reconstruidas.csv", index=False)
print("Archivos exportados en:", SALIDA.resolve())

## 6. Alcance de la reconstrucción

- **Verificado:** los CSV históricos, sus columnas y el aislamiento entre `split_train.csv` y `split_test_inicial.csv`.
- **Reconstruido, no verificado históricamente:** la asignación de pacientes del conjunto reservado a validación y prueba utilizando `GroupShuffleSplit` y semilla 42.
- **Pendiente:** comparar las series de validación/prueba con archivos originales, registros de ejecución o matrices finales para establecer si coinciden con el experimento publicado.
- **No inferir:** los conteos finales de *imágenes/cortes* a partir de estos conteos de *series*. La extracción de cortes se documentará por separado.

Si no es posible recuperar la asignación original, estos CSV solo deben utilizarse en un **experimento nuevo de reproducción**, no para atribuirles los resultados obtenidos anteriormente.